In [ ]:
from functools import partial

from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from covid import constants
from covid import feature
from covid.feature.column_dropper import ColumnDropper
from covid.feature.high_missing_rate_dropper import HighMissingRateDropper

full_preprocessor = Pipeline(
    [
        ("dropper", ColumnDropper(columns_to_drop=[feature.ID])),
        ("missing_rate_dropper", HighMissingRateDropper(missing_threshold=0.05)),
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ]
)

mutual_information = partial(
    mutual_info_classif, discrete_features=True, random_state=constants.RANDOM_STATE
)

selected_preprocessor = Pipeline(
    [
        ("dropper", ColumnDropper(columns_to_drop=[feature.ID])),
        ("missing_rate_dropper", HighMissingRateDropper(missing_threshold=0.05)),
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=mutual_information, k=5)),
    ]
)


In [ ]:
from umap import UMAP

umap_full = UMAP(
    n_neighbors=5,
    min_dist=0.01,
    n_components=2,
    random_state=constants.RANDOM_STATE
)

umap_full = Pipeline(
    [("preprocessor", full_preprocessor), ("reducer", umap_full)]
)

umap_selected = UMAP(
    min_dist=1.0,
    n_components=2,
    random_state=constants.RANDOM_STATE
)

umap_selected = Pipeline(
    [("preprocessor", selected_preprocessor), ("reducer", umap_selected)]
)

In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA(n_components=2, random_state=constants.RANDOM_STATE)
pca_full = Pipeline(
    [("preprocessor", full_preprocessor), ("reducer", pca_full)]
)

pca_selected = PCA(n_components=2, random_state=constants.RANDOM_STATE)
pca_selected = Pipeline(
    [("preprocessor", selected_preprocessor), ("reducer", pca_selected)]
)

In [ ]:
import pandas as pd

train_data = pd.read_csv(constants.INTERIM_TRAIN_DATA_PATH, dtype={feature.ID: str})
train_data.shape

In [ ]:
X_train = train_data.drop(columns=[feature.TARGET])
y_train = train_data[feature.TARGET]
X_train.shape, y_train.shape

In [ ]:
umap_full_emb = umap_full.fit_transform(X_train)
pca_full_emb = pca_full.fit_transform(X_train)

X_selected = umap_selected["preprocessor"].fit_transform(X_train, y_train)
umap_selected_emb = umap_selected["reducer"].fit_transform(X_selected)

X_selected = pca_selected["preprocessor"].fit_transform(X_train, y_train)
pca_selected_emb = pca_selected["reducer"].fit_transform(X_selected)

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 10))
axes = axes.flatten()

plots_data = [
    (axes[0], "UMAP Embeddings of Train Data", umap_full_emb),
    (axes[1], "UMAP Embeddings of Train Data (With Feature Selection)", umap_selected_emb),
    (axes[2], "PCA Embeddings of Train Data", pca_full_emb),
    (axes[3], "PCA Embeddings of Train Data (With Feature Selection)", pca_selected_emb),
]

for ax, title, emb in plots_data:
    ax.set_title(title)
    sns.scatterplot(
        x=emb[:, 0], y=emb[:, 1], ax=ax, hue=y_train
    )

plt.tight_layout()
plt.show()